# **CardiAg: participants sample descriptives**

Overview of the descriptive statistics of the participants sample for the CardiAg study, both before and after exclusions. In detail, we report: 
* Demographics for full sample before exclusion, including both complete and incomplete datasets (n=57)
* Demographics for full sample after exclusion, including only complete datasets (n=46)
* Demographics for the sub-sample included in the behavioral and cardiac analyses (n=44)
* Demographics for the sub-sample included in the respiratory and bimodal cardio-respiratory analyses (n=41)

Author: Marta Gerosa

Created on: 04 December 2024

Last update: 22 April 2026

In [1]:
############## Import modules ##############

import pandas as pd
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import ptitprince as pt
from IPython.display import Markdown as md

In [ ]:
############## Define path for BIDS participants files ##############

# Load the participants.tsv file
wd = r'.\data'              # change with data directory
exp_name = 'CardiAgIBTask'
subj_tsv = os.path.join(wd, 'participants.tsv')
subj_df = pd.read_csv(subj_tsv, sep='\t')

# Load the participants.json metadata file
subj_json = os.path.join(wd, 'participants.json')
with open(subj_json) as fjson:
    subj_info = json.load(fjson)

In [3]:
############## Get metadata from BIDS participants.json file ##############

# Extract map of levels for each variable
gender_map = subj_info['sex']['Levels']
handedness_map = subj_info['handedness']['Levels']
visual_aids_map = subj_info['visual_aids']['Levels']
included_map = subj_info['included_final']['Levels']
included_behecg_map = subj_info['included_beh_ecg']['Levels']
included_resp_map = subj_info['included_resp']['Levels']

# Map levels into datafram
subj_df['sex'] = subj_df['sex'].map(gender_map)
subj_df['handedness'] = subj_df['handedness'].map(handedness_map)
subj_df['visual_aids'] = subj_df['visual_aids'].astype(str).map(visual_aids_map)
subj_df['included_final'] = subj_df['included_final'].astype(str).map(included_map)
subj_df['included_beh_ecg'] = subj_df['included_beh_ecg'].astype(str).map(included_behecg_map)
subj_df['included_resp'] = subj_df['included_resp'].astype(str).map(included_resp_map)

In [4]:
############## Basic demographic calculations ##############

# Define with bp values to take: t1 if only one measurement, t2 if two measurement were made
subj_df['bp_sys_value'] = np.where(subj_df['bp_sys_t2'].notna(), subj_df['bp_sys_t2'], subj_df['bp_sys_t1'])
subj_df['bp_dia_value'] = np.where(subj_df['bp_dia_t2'].notna(), subj_df['bp_dia_t2'], subj_df['bp_dia_t1'])

# Define function to print basic demographic calculations
def calculate_demographics(df):
    total = len(df)
    avg_age = df['age'].mean()
    std_age = df['age'].std()
    min_age =  df['age'].min()
    max_age = df['age'].max()
    gender_distrib = df['sex'].value_counts()
    handedness_distrib = df['handedness'].value_counts()
    visual_aids_usage = df['visual_aids'].value_counts()
    avg_bp_sys = df['bp_sys_value'].mean()
    avg_bp_dia = df['bp_dia_value'].mean()
    
    return {
        "Total N Participants": total,
        "Mean Age": avg_age,
        "SD Age": std_age, 
        "Min Age": min_age, 
        "Max Age": max_age, 
        "Gender": gender_distrib.to_dict(),
        "Handedness": handedness_distrib.to_dict(), 
        "Use of Visual Aids": visual_aids_usage.to_dict(), 
        "Mean Systolic BP": avg_bp_sys, 
        "Mean Diastolic BP": avg_bp_dia
    }

In [5]:
############## Plot the participants distribution by age and gender ##############

# Define function to create bar plot divided by age and gender
def plot_age_gender(subj_df, title):

    # Define age bins and labels
    age_bins = [18, 20, 25, 30, 35, 40, 45]
    age_labels = ['18-20', '21-25', '26-30', '31-35', '36-40', '41-45']

    # Create a new column for age ranges
    subj_df.loc[:, 'age_range'] = pd.cut(subj_df['age'], bins=age_bins, labels=age_labels, right=True)

    # Create a pivot table by age range and gender
    age_gender_counts = subj_df.groupby(['age_range', 'sex'], observed=False).size().unstack(fill_value=0)

    # Plotting
    ax = age_gender_counts.plot(kind='bar', stacked=True, figsize=(8, 6), color=['#ff7f00', '#377eb8'], width=0.7)
    plt.title(title, fontsize=14)
    plt.xlabel('Age Range', fontsize=12)
    plt.ylabel('Number of Participants', fontsize=12)
    plt.xticks(rotation=0)
    plt.legend(title='Gender', loc='upper right')

    plt.tight_layout()
    plt.show()

## **Demographics for Full Sample before Exclusions**

Demographics for full sample before exclusion, including both complete and incomplete datasets (n=57).

In [6]:
# Print demographics for the entire sample, encompassing both complete and incomplete datasets
demographics_total = calculate_demographics(subj_df)

print("#####  Demographics for Full Sample before Exclusions (incl. both Complete and Incomplete Datasets)  #####\n")
for key, value in demographics_total.items():
    if isinstance(value, dict):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {round(value,2)}")

#####  Demographics for Full Sample before Exclusions (incl. both Complete and Incomplete Datasets)  #####

Total N Participants: 57
Mean Age: 29.18
SD Age: 7.55
Min Age: 18
Max Age: 44
Gender: {'Female': 30, 'Male': 27}
Handedness: {'Right': 57}
Use of Visual Aids: {'Yes': 29, 'No': 28}
Mean Systolic BP: 115.28
Mean Diastolic BP: 75.75


In [ ]:
plot_age_gender(subj_df, 'Participants by Age Range and Gender - Full Sample before Exclusions')

In [8]:
md("Before exclusions, including both complete and incomplete datasets, {} participants (mean age = {}, SD = {}; age range {}-{}; {} females, {} males) were recruited for the study. Their mean systolic/diastolic blood pressure was {}/{} mmHg.".format(
    demographics_total['Total N Participants'], 
    round(demographics_total['Mean Age'], 2), 
    round(demographics_total['SD Age'], 2), 
    demographics_total['Min Age'], 
    demographics_total['Max Age'], 
    demographics_total['Gender']['Female'], 
    demographics_total['Gender']['Male'], 
    round(demographics_total['Mean Systolic BP'], 2), 
    round(demographics_total['Mean Diastolic BP'], 2)
))

Before exclusions, including both complete and incomplete datasets, 57 participants (mean age = 29.18, SD = 7.55; age range 18-44; 30 females, 27 males) were recruited for the study. Their mean systolic/diastolic blood pressure was 115.28/75.75 mmHg.

## **Demographics for Final Sample after Exclusions**

Demographics for full sample after exclusion, including only complete datasets (n=46).

In [9]:
# Demographics for the final sample of included participants
included_subj = subj_df[subj_df['included_final'] == "Included"]
demographics_included = calculate_demographics(included_subj)

print("#####  Demographics for Final Sample after Exclusions (incl. only Complete Datasets)  #####\n")
for key, value in demographics_included.items():
    if isinstance(value, dict):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {round(value,2)}")

#####  Demographics for Final Sample after Exclusions (incl. only Complete Datasets)  #####

Total N Participants: 46
Mean Age: 28.54
SD Age: 7.23
Min Age: 18
Max Age: 44
Gender: {'Female': 25, 'Male': 21}
Handedness: {'Right': 46}
Use of Visual Aids: {'No': 24, 'Yes': 22}
Mean Systolic BP: 113.59
Mean Diastolic BP: 74.54


In [ ]:
plot_age_gender(included_subj, 'Participants by Age Range and Gender - Final Sample after Exclusions')

In [11]:
md("After exclusions, {} participants (mean age = {}, SD = {}; age range {}-{}; {} females, {} males) completed the experimental session and were included in the final sample. Their mean systolic/diastolic blood pressure was {}/{} mmHg.".format(
    demographics_included['Total N Participants'], 
    round(demographics_included['Mean Age'], 2), 
    round(demographics_included['SD Age'], 2), 
    demographics_included['Min Age'], 
    demographics_included['Max Age'], 
    demographics_included['Gender']['Female'], 
    demographics_included['Gender']['Male'], 
    round(demographics_included['Mean Systolic BP'], 2), 
    round(demographics_included['Mean Diastolic BP'], 2)
))

After exclusions, 46 participants (mean age = 28.54, SD = 7.23; age range 18-44; 25 females, 21 males) completed the experimental session and were included in the final sample. Their mean systolic/diastolic blood pressure was 113.59/74.54 mmHg.

In [ ]:
# Plot raincloud plots of systolic and diastolic BP by gender
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Systolic Blood Pressure (bp_sys_value)
pt.RainCloud(x='sex', y='bp_sys_value', data=included_subj, palette=['#ff7f00', '#377eb8'],
       bw=.3, width_viol=.5, width_box = .08, ax=axes[0], orient='v', point_size=4, linewidth=1.3, offset=.15)
axes[0].set_ylim([80,140])
axes[0].set_title('Systolic Blood Pressure by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Systolic Blood Pressure')

# Plot Diastolic Blood Pressure (bp_dia_value)
pt.RainCloud(x='sex', y='bp_dia_value', data=included_subj, palette=['#ff7f00', '#377eb8'],
       bw=.3, width_viol=.5, width_box = .08, ax=axes[1], orient='v', point_size=4, linewidth=1.3, offset=.15)
axes[1].set_ylim([50,100])
axes[1].set_title('Diastolic Blood Pressure by Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Diastolic Blood Pressure')
plt.tight_layout()

## **Demographics for Sub-Sample used in Behavioral and Cardiac Analyses**

Demographics for the sub-sample included in the behavioral and cardiac analyses (n=44), thus excluding participants with outlier JEs distributions from the intentional binding task (preregistered criterion) or corrupted ECG signal. 

In [13]:
# Demographics for the sub-sample used in behavioral and cardiac analyses
included_beh_ecg = subj_df[subj_df['included_beh_ecg'] == "Included"]
demographics_beh_ecg = calculate_demographics(included_beh_ecg)

print("#####  Demographics for Sub-Sample used in Behavioral and Cardiac Analyses  #####\n")
for key, value in demographics_beh_ecg.items():
    if isinstance(value, dict):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {round(value,2)}")

#####  Demographics for Sub-Sample used in Behavioral and Cardiac Analyses  #####

Total N Participants: 44
Mean Age: 28.7
SD Age: 7.24
Min Age: 18
Max Age: 44
Gender: {'Female': 23, 'Male': 21}
Handedness: {'Right': 44}
Use of Visual Aids: {'Yes': 22, 'No': 22}
Mean Systolic BP: 113.68
Mean Diastolic BP: 74.61


In [14]:
# Demographics for the sub-sample used in respiratory and bimodal cardio-respiratory analyses
included_resp = subj_df[subj_df['included_resp'] == "Included"]
demographics_resp = calculate_demographics(included_resp)

print("#####  Demographics for Sub-Sample used in Respiratory and Bimodal Cardio-Respiratory Analyses  #####\n")
for key, value in demographics_resp.items():
    if isinstance(value, dict):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {round(value,2)}")

#####  Demographics for Sub-Sample used in Respiratory and Bimodal Cardio-Respiratory Analyses  #####

Total N Participants: 41
Mean Age: 27.95
SD Age: 6.69
Min Age: 18
Max Age: 43
Gender: {'Female': 22, 'Male': 19}
Handedness: {'Right': 41}
Use of Visual Aids: {'Yes': 21, 'No': 20}
Mean Systolic BP: 113.07
Mean Diastolic BP: 74.32
